In [1]:
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

project_root = Path("../../..").resolve()
sys.path.append(str(project_root))

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool

from utils import DataManager
from utils.plotter import actual_vs_prediction
torch.manual_seed(25)
np.random.seed(25)


c:\Users\Clent\miniconda3\envs\quantum-classical\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
df = DataManager('admet/mutagenicity', dataset_type='graph').dataset

train_dataset, test_dataset = train_test_split(df, test_size=0.2, random_state=25)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)    

In [4]:
class GINRegressor(nn.Module):
    def __init__(
        self,
        in_channels, 
        out_channels, 
        hidden_feat,
        num_layers,
    ):
        super().__init__()

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(in_channels, 2 * hidden_feat),
                nn.BatchNorm1d(2 * hidden_feat),
                nn.ReLU(),
                nn.Linear(hidden_feat * 2, hidden_feat)

            )
            conv = GINConv(mlp, train_eps=True)

            self.convs.append(conv)
            self.norms.append(nn.BatchNorm1d(hidden_feat))
            in_channels = hidden_feat

        self.fc1 = nn.Linear(hidden_feat, hidden_feat // 2 )
        self.fc2 = nn.Linear(hidden_feat // 2, out_channels)
        self.norms1 = nn.BatchNorm1d(hidden_feat // 2)

    def _encode(self, x, edge_index):
        for convs, norms in zip(self.convs, self.norms):
            x = convs(x, edge_index)
            x = F.relu(norms(x))
        
        return x
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch 

        h = self._encode(x, edge_index)
        h = global_mean_pool(h, batch)
        h = F.relu(self.norms1(self.fc1(h)))
        h = self.fc2(h)
        
        return h    

In [5]:
model = GINRegressor(in_channels=6, out_channels=2, hidden_feat=64, num_layers=2).to(device)
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=15
)
criterion = nn.CrossEntropyLoss()

In [6]:
def train(model, optimizer, criterion, loader):
    model.train()
    total_loss = 0

    for data in loader:
        data = data.to(device)

        optimizer.zero_grad()

        # Model output: [num_graphs, num_classes]
        out = model(data)

        # Class labels: [num_graphs]
        y = data.y.view(-1).long()

        loss = criterion(out, y)
        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        total_loss += loss.item() * data.num_graphs

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    y_true = []
    y_pred = []

    for data in loader:
        data = data.to(device)

        # Model output: [num_graphs, num_classes]
        out = model(data)

        # Get predicted class
        pred = out.argmax(dim=1)

        y_true.extend(
            data.y.view(-1).cpu().numpy()
        )

        y_pred.extend(
            pred.view(-1).cpu().numpy()
        )

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    return {
        "y_true": y_true,
        "y_pred": y_pred,

        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "precision": precision_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "confusion_matrix": confusion_matrix(
            y_true,
            y_pred
        )
    }

In [7]:

best_test_f1 = -np.inf
best_state = None

patience_counter = 0
early_stop_patience = 40


for epoch in range(500):

    # =========================
    # Training
    # =========================
    train_loss = train(
        model,
        optimizer,
        criterion,
        train_loader
    )

    # =========================
    # Evaluate
    # =========================
    test_metrics = evaluate(
        model,
        test_loader
    )

    # Scheduler monitors training loss
    scheduler.step(train_loss)

    # =========================
    # Save best model
    # =========================
    if test_metrics["f1"] > best_test_f1:

        best_test_f1 = test_metrics["f1"]

        best_state = {
            k: v.cpu().clone()
            for k, v in model.state_dict().items()
        }

        patience_counter = 0

    else:
        patience_counter += 1

    # =========================
    # Print progress
    # =========================
    if (epoch + 1) % 10 == 0:

        train_metrics = evaluate(
            model,
            train_loader
        )

        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch+1:03d} | "
            f"Loss {train_loss:.4f} | "
            f"Train Acc {train_metrics['accuracy']:.4f} | "
            f"Test Acc {test_metrics['accuracy']:.4f} | "
            f"Test F1 {test_metrics['f1']:.4f} | "
            f"LR {current_lr:.2e}"
        )

    # =========================
    # Early stopping
    # =========================
    if patience_counter >= early_stop_patience:

        print(
            f"Early stopping at epoch {epoch+1} "
            f"(best test F1={best_test_f1:.4f})"
        )

        break


# =========================
# Restore best model
# =========================

model.load_state_dict(best_state)


# =========================
# Final test evaluation
# =========================

final_metrics = evaluate(
    model,
    test_loader
)

print("\nFinal (best) test metrics:")
print(f"Accuracy : {final_metrics['accuracy']:.4f}")
print(f"Precision: {final_metrics['precision']:.4f}")
print(f"Recall   : {final_metrics['recall']:.4f}")
print(f"F1 Score : {final_metrics['f1']:.4f}")

print("\nConfusion Matrix:")
print(final_metrics["confusion_matrix"])

Epoch 010 | Loss 0.4945 | Train Acc 0.7777 | Test Acc 0.7736 | Test F1 0.7744 | LR 1.00e-03
Epoch 020 | Loss 0.4472 | Train Acc 0.7936 | Test Acc 0.7335 | Test F1 0.7334 | LR 1.00e-03
Epoch 030 | Loss 0.3928 | Train Acc 0.8555 | Test Acc 0.7618 | Test F1 0.7609 | LR 1.00e-03
Epoch 040 | Loss 0.3665 | Train Acc 0.8443 | Test Acc 0.7665 | Test F1 0.7630 | LR 1.00e-03
Epoch 050 | Loss 0.3024 | Train Acc 0.8838 | Test Acc 0.7736 | Test F1 0.7737 | LR 1.00e-03
Early stopping at epoch 59 (best test F1=0.7963)

Final (best) test metrics:
Accuracy : 0.7972
Precision: 0.7967
Recall   : 0.7972
F1 Score : 0.7963

Confusion Matrix:
[[137  49]
 [ 37 201]]


In [12]:
print(classification_report(final_metrics['y_true'], final_metrics['y_pred']))  

              precision    recall  f1-score   support

         0.0       0.79      0.74      0.76       186
         1.0       0.80      0.84      0.82       238

    accuracy                           0.80       424
   macro avg       0.80      0.79      0.79       424
weighted avg       0.80      0.80      0.80       424



In [15]:
@torch.no_grad()
def predict(model, loader, device="cuda"):
    model.eval()

    predictions = []

    for data in loader:
        data = data.to(device)

        out = model(data)

        # Classification
        pred = out.argmax(dim=1)

        predictions.extend(pred.cpu().numpy())

    return predictions

In [ ]:
inf_df = DataManager('roselle_dataset', dataset_type='graph').dataset 
inf_loader = DataLoader(inf_df, batch_size=32, shuffle=True)
 

In [ ]:
import pandas as pd
inf_df = DataManager(
    'roselle_dataset',
    dataset_type='graph'
).dataset

inf_df_final = DataManager(
    'roselle_dataset'
).dataset

inf_loader = DataLoader(
    inf_df,
    batch_size=32,
    shuffle=False
)

predictions = predict(
    model,
    inf_loader,
    device=device
)

pred_df = pd.DataFrame(predictions, columns=['mutagenicity'])

final_dataset = pd.concat(objs=[inf_df_final, pred_df], axis=1)


,CID,MolecularFormula,MolecularWeight,smiles,ConnectivitySMILES,InChIKey,IUPACName,HBondDonorCount,HBondAcceptorCount,mol_weight,...,fp_1015,fp_1016,fp_1017,fp_1018,fp_1019,fp_1020,fp_1021,fp_1022,fp_1023,mutagenicity
0,311,C6H8O7,192.12,C(C(=O)O)C(CC(=O)O)(C(=O)O)O,C(C(=O)O)C(CC(=O)O)(C(=O)O)O,KRKNYBCHXYNGOX-UHFFFAOYSA-N,"2-hydroxypropane-1,2,3-tricarboxylic acid",4,7,192.123,...,0,0,0,0,0,0,0,0,0,0
1,61300,C6H5FeO7,244.94,C(C(=O)[O-])C(CC(=O)[O-])(C(=O)[O-])O.[Fe+3],C(C(=O)[O-])C(CC(=O)[O-])(C(=O)[O-])O.[Fe+3],NPFOYSMITVOQOS-UHFFFAOYSA-K,"2-hydroxypropane-1,2,3-tricarboxylate;iron(3+)",1,7,244.944,...,0,0,0,0,0,0,0,0,1,0
2,2735208,C6H7K3O8,324.41,C(C(=O)[O-])C(CC(=O)[O-])(C(=O)[O-])O.O.[K+].[...,C(C(=O)[O-])C(CC(=O)[O-])(C(=O)[O-])O.O.[K+].[...,PJAHUDTUZRZBKM-UHFFFAOYSA-K,"tripotassium;2-hydroxypropane-1,2,3-tricarboxy...",2,8,324.408,...,0,0,0,0,0,0,0,0,1,0
3,1198,C6H8O7,192.12,C(C(C(C(=O)O)O)C(=O)O)C(=O)O,C(C(C(C(=O)O)O)C(=O)O)C(=O)O,ODBLHEXUDAPZAU-UHFFFAOYSA-N,"1-hydroxypropane-1,2,3-tricarboxylic acid",4,7,192.123,...,0,0,0,0,0,0,0,0,0,0
4,31348,C6H5O7-3,189.10,C(C(=O)[O-])C(CC(=O)[O-])(C(=O)[O-])O,C(C(=O)[O-])C(CC(=O)[O-])(C(=O)[O-])O,KRKNYBCHXYNGOX-UHFFFAOYSA-K,"2-hydroxypropane-1,2,3-tricarboxylate",1,7,189.099,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1279,44567203,C27H42O5,446.60,C[C@@H]1CC[C@@]2([C@H]([C@H]3[C@@H](O2)C[C@@]4...,CC1CCC2(C(C3C(O2)CC4(C3(CCC5C4CC=C6C5(C(CC(C6)...,UCMNVQWKZHALAW-CQQWHIIOSA-N,"(1R,2R,4S,5'R,6R,7S,8R,9R,12S,13R,14R,16R)-5',...",3,5,446.628,...,0,0,1,0,1,0,0,0,0,0
1280,44575244,C50H80O23,1049.20,C[C@H]1[C@H]2[C@H](C[C@@H]3[C@@]2(CC[C@H]4[C@H...,CC1CCC2(C(C3C(O2)CC4C3(CCC5C4CC=C6C5(CCC(C6)OC...,IMUPDJRJGQSNAI-DVZGRTHJSA-N,"(2R,3R,4R,5S,6S)-6-[(2S,3R,4S,5R,6R)-2-[(2R,3R...",13,23,1049.167,...,0,0,1,0,1,0,0,0,0,1
1281,44575742,C46H74O18,915.10,C[C@@H]1[C@H](O[C@H]([C@@H]([C@H]1O)O[C@H]2[C@...,CC1C(OC(C(C1O)OC2C(C(C(C(O2)CO)O)O)O)OC3C(OC(C...,YKLQFJHOYKPXLL-XEGGNEBNSA-N,"(2S,3R,4S,5S,6R)-2-[(2S,3R,4S,5S,6S)-2-[(2R,3R...",10,18,915.080,...,0,0,1,0,1,0,0,0,0,1
1282,44575743,C44H70O16,855.00,C[C@@H]1CC[C@@]2([C@H]([C@H]3[C@@H](O2)C[C@@H]...,CC1CCC2(C(C3C(O2)CC4C3(CCC5C4CC=C6C5(CCC(C6)OC...,HSOMTBUZSIVDQK-OJQSNVAUSA-N,"(2S,3R,4R,5R,6S)-2-[(2R,3R,4S,5S,6R)-4-hydroxy...",8,16,855.028,...,0,0,1,0,1,0,0,0,0,1


In [37]:
output = project_root / 'data' / 'final_dataset'
output.mkdir(parents=True, exist_ok=True)

output_file = output / 'final_roselle_dataset.csv'
final_dataset.to_csv(output_file, index=False)